# Causality-Aware Graph Neural Networks

## Prerequisites

First, we need to set up our Python environment that has PyTorch, PyTorch Geometric and PathpyG installed. Depending on where you are executing this notebook, this might already be (partially) done. E.g. Google Colab has PyTorch installed by default so we only need to install the remaining dependencies. The DevContainer that is part of our GitHub Repository on the other hand already has all of the necessary dependencies installed.

In the following, we install the packages for usage in Google Colab using Jupyter magic commands. For other environments comment in or out the commands as necessary. For more details on how to install `pathpyG` especially if you want to install it with GPU-support, we refer to our [documentation](https://www.pathpy.net/dev/getting_started/). Note that `%%capture` discards the full output of the cell to not clutter this tutorial with unnecessary installation details. If you want to print the output, you can comment `%%capture` out.

## Motivation and Learning Objectives

In previous tutorials, we have introduced causal paths in temporal graphs, and how we can use them to generate higher-order De Bruijn graph models that capture temporal-topological patterns in time series data. In this tutorial, we will show how we can use De Bruijn Graph Neural Networks, a causality-aware deep learning architecture for temporal graph data. The details of this approach are introduced [in this paper](https://proceedings.mlr.press/v198/qarkaxhija22a.html). The architecture is implemented in pathpyG and can be readily applied to temporal graph data.

Below we illustrate this method in a supervised node classification task, i.e. given a temporal graph we will use the temporal-topological patterns in the graph to classify nodes.

We start by importing a few modules:

In [ ]:
import os
import tempfile
from copy import deepcopy
from urllib import request

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import torch
from sklearn.manifold import TSNE
from sklearn.metrics import balanced_accuracy_score
from torch_geometric.transforms import RandomNodeSplit

import pathpyG as pp
from pathpyG.nn.dbgnn import DBGNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Temporal-Topological Clusters in Temporal Graphs

Let us load a small synthetic toy example for a temporal graph with 60.000 time-stamped interactions between 30 nodes. We use the `TemporalGraph` class to load this example from a file containing edges with discrete time-stamps.

<div class="admonition note">
    <p class="admonition-title">Dataset Availability</p>
    <p>
        Depending on how you are executing this notebook, you may need to download the dataset first. We automatically check if the dataset is available in the relative path <code>../data/temporal_clusters.tedges</code>, which is the default location if you cloned the pathpyG repository. If the file is not found, we download it from the GitHub repository.
    </p>
</div>

In [ ]:
if os.path.exists('../data/temporal_clusters.tedges'):
    print("Loading dataset from local path...")
    t = pp.io.read_csv_temporal_graph('../data/temporal_clusters.tedges', header=False)
else:
    print("Loading dataset from remote URL...")
    with tempfile.TemporaryDirectory() as tmpdir:
        url = "https://raw.githubusercontent.com/pathpy/pathpyG/refs/heads/main/docs/data/temporal_clusters.tedges"
        file_path = os.path.join(tmpdir, 'temporal_clusters.tedges')
        request.urlretrieve(url, file_path)
        t = pp.io.read_csv_temporal_graph(file_path, header=False)

t = t.to(device)

This example has created in such a way that the nodes naturally form three clusters, which are highlighted in the interactive visualization below:

In [ ]:
style = {}
style["node_color"] = ["green"] * 10 + ["red"] * 10 + ["blue"] * 10
pp.plot(t, **style, show_labels=False);

## Modelling Causal Structures with Higher-Order De Bruijn Graphs

But what is the origin for the cluster pattern? In the visualization above, you will notice that the time-stamped edges randomly interconnect nodes within and across clusters, actually there is no correlation whatsoever between the topology of links and the cluster membership of the nodes. Hence, the notion of clusters does not correspond to the common idea of cluster patterns in static graphs, which we can highlight further by plotting the static time-aggregated network:

In [ ]:
pp.plot(t.to_static_graph(), **style, show_labels=False);

In fact, the topology of this graph corresponds to that of a random graph, i.e. there are not patterns whatsoever in the topology of links. Nevertheless, the temporal graph contains a cluster pattern in the topology of causal or time-respecting paths. In particular, the temporal ordering of time-stamped edges is such that nodes with the same cluster label are more frequently connected by time-respecting paths than nodes with different cluster labels. Hence, nodes within the same clusters can more strongly influence each other in a causal way, i.e. via multiple interactions that follow the arrow of time.

Traditional (temporal) graph neural networks will not be able to learn from this pattern, as it is due to the specific microscopic temporal ordering of edges. Using higher-order De Bruijn graph models implemented in pathpyG, we can learn from temporal graph data that contains such patterns. Let us explain this step by step.

Referring to the previous tutorial on causal paths in temporal graphs, we first create a node-time directed acyclic graph that captures the causal structure of the temporal graph. In this small example, we will only consider two time-stamped edges $(u,v;t)$ and $(v,w;t')$ to contribute to a causal path iff $0 < t'-t \leq 1$, i.e. we use a delta for the maximum time difference of one time step.

In [ ]:
%%capture
m = pp.MultiOrderModel.from_temporal_graph(t, max_order=2)

In [ ]:
print(m)

We can get the first and second order networks from the Multi Order Network object. The first order network is the network of nodes and edges, while the second order network is the network of first order edges as second order nodes and second order edges. The second order network is a De Bruijn graph that captures the temporal-topological patterns in the data.

In [ ]:
g = m.layers[1]
g2 = m.layers[2]

In [ ]:
pp.plot(g, edge_size=1, show_labels=False);

Since it does not consider patterns in the causal topology of the temporal graph, this is not a meaningful model. We can instead use a second-order De Bruijn graph model, which we can easily fit to the paths:

In [ ]:
layout_style = {}
layout_style["layout"] = "Fruchterman-Reingold"
layout_style["seed"] = 1
layout_style["k"] = 0.5
layout_style["iterations"] = 300
layout = pp.layout(g2, **layout_style)
pp.plot(g2, backend="matplotlib", layout=layout, edge_size=0.5, node_size=3, show_labels=False);

In this graph, every node is a link and links correspond to causal paths of length two, i.e. temporally ordered sequences consisting of two edges that overlap in the center node. In this graph, we clearly see a cluster pattern that is due to the way in which temporal edges are ordered in time. In particular, we see three clusters, where the edges in three of the clusters correspond to causal paths of length two that connect nodes within each of the three clusters. The edges in the fourth cluster (in the center of the visualization) represent causal paths that connect nodes in different clusters.

## Comparison to Temporal Graph with Shuffled Time Stamps

You may wonder whether this pattern is really due to the temporal ordering of time-stamped edges. It is easy to check this. We can simply randomly shuffle the time stamps of all edges, which will break any correlations in the temporal ordering that lead to patterns in the causal topology.

We repeat the path calculation for this shuffled temporal graph and construct the second-order De Bruijn Graph model again:

In [ ]:
t_shuffled = deepcopy(t)
t_shuffled.shuffle_time()

In [ ]:
%%capture
g2_shuffled = pp.MultiOrderModel.from_temporal_graph(t_shuffled, max_order=2).layers[2]

In [ ]:
print(g2_shuffled)

In [ ]:
layout = pp.layout(g2_shuffled, **layout_style)
pp.plot(g2_shuffled, backend="matplotlib", layout=layout, edge_size=0.5, node_size=3, show_labels=False);

We now find that the cluster pattern in the second-order graph has vanished. In fact, there is no pattern whatsoever since the underlying (static) graph topology is random and the random shuffling of time stamps leads to random causal paths.

## Spectral clustering with second-order graph Laplacian

To take a different perspective on cluster patterns, we can actually use `pathpyG` to apply a spectral analysis to the higher-order graph. We can simply calculate a generalization of the Laplacian matrix to the second-order graph both for the actual temporal graph and its shuffled counterpart:

In [ ]:
L = g2.laplacian(normalization='rw', edge_attr='edge_weight')
L_shuffled= g2_shuffled.laplacian(normalization='rw',edge_attr='edge_weight')

We then calculate the eigenvalues and eigenvectors of the Laplacians, and compute the Fiedler vector, i.e. the eigenvector that corresponds to the second-smallest eigenvalue of the Laplacian.

In [ ]:
w,v = sp.linalg.eig(L.todense(),left= False, right = True)
w_shuffled, v_shuffled = sp.linalg.eig(L_shuffled.todense())

In [ ]:
fiedler = v[:,np.argsort(w)[1]]
fiedler_shuffled = v_shuffled[:,np.argsort(w_shuffled)[1]]

Below, we show that the clusters in the causal topology of the temporal graph correspond to clusters in the distribution of entries in the Fiedler vector, while there is no such pattern for the Fiedler vector of the second-order graph constructed from the shuffled temporal graph:

In [ ]:
def higher_order_class_assignment(ho_node_id):
    """Assign class labels based on the higher-order node ids.

    There are three classes that were assigned based on the original node ids:
        - Class 0: node ids 0-9
        - Class 1: node ids 10-19
        - Class 2: node ids 20-29

    The higher-order patterns were constructed such that the clusters are formed by second-order nodes whose first-order node ids belong to the same class.
    We therefore assign higher-order class labels based on the first-order node ids in the higher-order node id tuple.
    """
    if ho_node_id[0] < 10 and ho_node_id[1] < 10:
        return 0
    elif ho_node_id[0] < 20 and ho_node_id[0] >= 10 and ho_node_id[1] < 20 and ho_node_id[1] >= 10:
        return 1
    elif ho_node_id[0] < 30 and ho_node_id[0] >= 20 and ho_node_id[1] < 30 and ho_node_id[1] >= 20:
        return 2
    else:
        return 3

In [ ]:
colors = {0: 'green', 1: 'red', 2: 'blue', 3: 'gray'}
opacities = {0: 0.6, 1: 0.6, 2: 0.6, 3: 0.1}

In the plots below, we have colored those entries of the Fiedler vectors that correspond to edges connecting nodes within one of the three clusters shown above. The Fiedler vector shows a clear pattern, which translates to the cluster pattern in the causal topology that we have planted into our synthetic temporal graph.

In [ ]:
ho_class_ids = list(map(higher_order_class_assignment, g2.nodes))
plt.scatter(
    range(g2.n), np.real(fiedler), c=[colors[i] for i in ho_class_ids], alpha=[opacities[i] for i in ho_class_ids]
)
plt.ylim(-0.25, 0.25)

No such pattern exists in the Fiedler vector of the second-order graph corresponding to the shuffled `TemporalGraph`.

In [ ]:
shuffled_ho_class_ids = list(map(higher_order_class_assignment, g2_shuffled.nodes))
plt.scatter(
    range(g2_shuffled.n),
    np.real(fiedler_shuffled),
    c=[colors[i] for i in shuffled_ho_class_ids],
    alpha=[opacities[i] for i in shuffled_ho_class_ids],
)
plt.ylim(-0.25, 0.25)

## Node Classification with Causality-Aware Graph Neural Networks

Let us now explore how we can develop a causality-aware deep graph learning architecture that utilizes this pattern in the causal topology. We will follow the architecture introduced [in this work](https://proceedings.mlr.press/v198/qarkaxhija22a.html). The architecture actually performs message passing in higher-order models with multiple orders at once. In a final message passing step, a bipartite graph is used to obtain vector-space representations of actual nodes in the temporal graph.

We now set up a `pytorch_geometric.Data` object that contains all of the information needed to train the DBGNN model. For this, we can use a convenience function of the `MultiOrderModel` class in `pathpyG`. Combining a first- and a second-order model, this uses the edge indices and the weight tensors for a message passing scheme. it further constructs an `edge_index` of a bipartite graph that uses the last node in a second-order node to map messages back to first-order nodes.

In [ ]:
data = m.to_dbgnn_data(max_order=2, mapping="last")
data.y = torch.tensor([int(i) // 10 for i in t.nodes], device=device)

## Training the model

We are now ready to train and evaluate our causality-aware graph neural network. We will frist create a random split of the nodes, set the optimizer and the hyperparameters of our model.

In [ ]:
data = RandomNodeSplit(num_val=0, num_test=0.3)(data)

model = DBGNN(num_features=[g.n, g2.n], num_classes=len(data.y.unique()), hidden_dims=[16, 32, 8], p_dropout=0.4).to(
    device
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_function = torch.nn.CrossEntropyLoss()

The following function evaluates the prediction of our model based on the balanced accuracy score for categorical predictions.

In [ ]:
def test(model, data):
    """Evaluate the model on training and test data."""
    model.eval()

    _, pred = model(data).max(dim=1)

    metrics_train = balanced_accuracy_score(data.y[data.train_mask].cpu(), pred[data.train_mask].cpu().numpy())

    metrics_test = balanced_accuracy_score(data.y[data.test_mask].cpu(), pred[data.test_mask].cpu().numpy())

    return metrics_train, metrics_test

In [ ]:
losses = []
for epoch in range(50):
    output = model(data)
    loss = loss_function(output[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    losses.append(loss)

    if epoch % 10 == 0:
        train_ba, test_ba = test(model, data)
        print(f"Epoch: {epoch}, Loss: {loss}, Train balanced accuracy: {train_ba}, Test balanced accuracy: {test_ba}")

# Causality-aware latent space representation of nodes

We can inspect the model by plotting a latent space representation of the edges generated by the second-order layer of our architecture.

In [ ]:
model.eval()
latent = model.higher_order_layers[0].forward(data.x_h, data.edge_index_higher_order).detach()
latent = model.higher_order_layers[1].forward(latent, data.edge_index_higher_order).detach()
node_embedding = TSNE(n_components=2, learning_rate="auto", init="random").fit_transform(latent.cpu())

embedding_layout = {v: node_embedding[g2.mapping.to_idx(v)] for v in g2.nodes}
pp.plot(g2, backend="matplotlib", layout=embedding_layout, show_labels=False, edge_size=0.3, node_size=3, edge_opacity=0.1, node_color=[colors[i] for i in ho_class_ids]);

We can further generate latent space representations of the nodes generated by the last bipartite layer of our architecture:

In [ ]:
model.eval()
latent = model.forward(data).detach()
node_embedding = TSNE(n_components=2, learning_rate='auto', init='random', perplexity=10).fit_transform(latent.cpu())

embedding_layout = {v: node_embedding[g.mapping.to_idx(v)] for v in g.nodes}
pp.plot(g, backend="matplotlib", layout=embedding_layout, show_labels=False, edge_size=0.3, node_size=3, edge_opacity=0.1, **style);

# Explainer evaluation: model-ground-truth + generator-ground-truth

This section evaluates hierarchical explanations at two levels:

**Level 1 (HO nodes):** explain a node \(v\) via incoming higher-order nodes \((u,v)\) that map to \(v\) in the bipartite step.

**Level 2 (HO edges):** explain the embedding of an HO node \((u,v)\) via higher-order edges \((u,v)\rightarrow(v,w)\) in the 2nd-order De Bruijn graph.

We compute two kinds of "ground truth":

1. **Model-ground-truth (faithfulness):** importance under the *trained model*.
   - Level 1: Shapley values over incoming HO nodes for a node \(v\) (permutation approximation).
   - Level 2: leave-one-out (LOO) over HO edges in a local \(k\)-hop neighborhood for an HO node \((u,v)\).

2. **Generator-ground-truth (planted mechanism):** importance w.r.t. the planted clusters.
   - Level 1 HO node \((u,v)\) is "signal" if `y[u]==y[v]`.
   - Level 2 HO edge \((u,v)\rightarrow(v,w)\) is "signal" if `y[u]==y[v]==y[w]`.

**Tip:** If the predicted class does not change when removing edges, use *margin drop* and embedding/logit changes as the evaluation target (not only label flips).


In [ ]:

# --- Parameters (edit these) ---
target_node = 19          # original node v to explain
top_m = 5                 # how many incoming HO nodes (u,v) to evaluate at Level 2
k_hop = 1                 # k-hop neighborhood in the HO graph for Level 2
lvl1_permutations = 2000  # Shapley permutation samples (Level 1 model-ground-truth)
ig_steps = 30             # integrated gradients steps (Level 2 explainer)
top_e = 10                # number of HO edges to print per HO node

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

# Use existing device if defined earlier in the notebook
device = globals().get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# Sanity checks: we assume you ran training cells above
assert "model" in globals(), "Expected a trained `model` to exist. Please run the training cells above."
assert "data" in globals(), "Expected `data` to exist. Please run the data construction cells above."
model = model.to(device).eval()
data = data.to(device)

# Pull required tensors from `data` (with fallbacks)
edge_index_fo = getattr(data, "edge_index")
edge_weight_fo = getattr(data, "edge_weight", None)

edge_index_ho = getattr(data, "edge_index_higher_order")
edge_weight_ho = getattr(data, "edge_weight_higher_order", None)

bip_edge_index = getattr(data, "bipartite_edge_index")

# Fallbacks if weights are missing (should not happen for to_dbgnn_data)
if edge_weight_fo is None:
    edge_weight_fo = torch.ones(edge_index_fo.size(1), device=device)
if edge_weight_ho is None:
    edge_weight_ho = torch.ones(edge_index_ho.size(1), device=device)

# Labels (generator ground truth)
y = data.y.detach().cpu().numpy()
num_nodes = int(data.y.numel())

# --- Build mapping HO node index -> (u,v) ---
# Preferred: use the `g2` object from earlier cells (it stores HO nodes as tuples)
g2_node_ids = None
if "g2" in globals():
    g2_node_ids = np.zeros((g2.n, 2), dtype=int)
    for ho_node in g2.nodes:
        idx = g2.mapping.to_idx(ho_node)
        g2_node_ids[idx, 0] = int(ho_node[0])
        g2_node_ids[idx, 1] = int(ho_node[1])
else:
    # Fallback: load from file if present
    import os
    if os.path.exists("g2_node_ids.npy"):
        g2_node_ids = np.load("g2_node_ids.npy")
    else:
        raise RuntimeError("Could not build HO node mapping. Expected `g2` or `g2_node_ids.npy`.")

num_ho_nodes = g2_node_ids.shape[0]
E_ho = int(edge_index_ho.size(1))

# --- Build adjacency lists for HO graph (for k-hop neighborhoods) ---
src = edge_index_ho[0].detach().cpu().numpy()
dst = edge_index_ho[1].detach().cpu().numpy()
adj_out = [[] for _ in range(num_ho_nodes)]
adj_in  = [[] for _ in range(num_ho_nodes)]
for s, d in zip(src, dst):
    adj_out[int(s)].append(int(d))
    adj_in[int(d)].append(int(s))

def k_hop_nodes(seed: int, k: int, direction: str = "both"):
    # Return set of HO node indices within k hops of seed.
    visited = {int(seed)}
    frontier = {int(seed)}
    for _ in range(k):
        nxt = set()
        for u in frontier:
            if direction in ("out", "both"):
                nxt.update(adj_out[u])
            if direction in ("in", "both"):
                nxt.update(adj_in[u])
        nxt -= visited
        if not nxt:
            break
        visited |= nxt
        frontier = nxt
    return visited

def edges_induced_by_nodes(node_set: set[int]):
    # Return HO edge ids whose endpoints are both in node_set.
    node_mask = np.zeros(num_ho_nodes, dtype=bool)
    node_mask[list(node_set)] = True
    e_mask = node_mask[src] & node_mask[dst]
    return np.where(e_mask)[0]

# --- Forward pieces of DBGNN so we can do fast interventions ---
def fo_embed():
    x = data.x
    x = F.elu(model.first_order_layers[0](x, edge_index_fo, edge_weight_fo))
    x = F.elu(model.first_order_layers[1](x, edge_index_fo, edge_weight_fo))
    return x

def ho_embed(edge_w_ho: torch.Tensor):
    x_h = data.x_h
    x_h = F.elu(model.higher_order_layers[0](x_h, edge_index_ho, edge_w_ho))
    x_h = F.elu(model.higher_order_layers[1](x_h, edge_index_ho, edge_w_ho))
    return x_h

def bipartite_aggregate(H: torch.Tensor, Xfo: torch.Tensor):
    # Re-implement BipartiteGraphOperator: message = x_i + x_j, aggregate to FO nodes.
    xh_t = model.bipartite_layer.lin1(H)   # (N_ho, d_out)
    xf_t = model.bipartite_layer.lin2(Xfo) # (N, d_out)

    src_b, dst_b = bip_edge_index[0], bip_edge_index[1]
    out_ho = torch.zeros((num_nodes, xh_t.size(1)), device=device, dtype=xh_t.dtype)
    out_ho.index_add_(0, dst_b, xh_t[src_b])

    deg = torch.bincount(dst_b, minlength=num_nodes).to(out_ho.dtype).unsqueeze(1)
    out = out_ho + deg * xf_t
    return out, out_ho, deg, xf_t, xh_t

def logits_from_out(out: torch.Tensor):
    node_emb = F.elu(out)
    return model.lin(node_emb)

def margin_of_node(logits: torch.Tensor, v: int, y_ref: int):
    row = logits[v]
    mask = torch.ones(row.numel(), dtype=torch.bool, device=row.device)
    mask[y_ref] = False
    return row[y_ref] - row[mask].max()

# Compute full forward once (for reuse)
with torch.no_grad():
    Xfo_full = fo_embed()
    H_full = ho_embed(edge_weight_ho)
    out_full, out_ho_full, deg_full, xf_t_full, xh_t_full = bipartite_aggregate(H_full, Xfo_full)
    logits_full = logits_from_out(out_full)
    pred_full = logits_full.argmax(dim=1)
    y_ref = int(pred_full[target_node].item())
    margin_full = float(margin_of_node(logits_full, target_node, y_ref).item())

print("Full prediction for node", target_node, "=", y_ref, "| full margin =", margin_full)

# Incoming HO nodes q=(u,target_node) are those with bipartite dst == target_node
src_b_np = bip_edge_index[0].detach().cpu().numpy()
dst_b_np = bip_edge_index[1].detach().cpu().numpy()
incoming_ho = src_b_np[dst_b_np == target_node].astype(int)
incoming_uv = g2_node_ids[incoming_ho]  # (u,v) pairs

print("Incoming HO nodes for v =", target_node, ":", len(incoming_ho))
print("First 10 (HO_idx, (u,v)):", list(zip(incoming_ho[:10], map(tuple, incoming_uv[:10]))))


In [ ]:

# ============================================================
# Level 1 evaluation: incoming HO nodes (u, v) for target node
# ============================================================

import numpy as np

M = len(incoming_ho)
assert M > 0

# Helper: compute margin from an "out" vector for ONE node v (fast)
W_lin = model.lin.weight.detach()  # (C, d)
b_lin = model.lin.bias.detach()

def logits_from_out_v(out_v: torch.Tensor):
    emb_v = F.elu(out_v)
    return (W_lin @ emb_v) + b_lin  # (C,)

def margin_from_out_v(out_v: torch.Tensor, y_ref: int):
    logits_v = logits_from_out_v(out_v)
    other = torch.cat([logits_v[:y_ref], logits_v[y_ref+1:]])
    return float(logits_v[y_ref] - other.max())

# Precompute contributions of each incoming HO node to v
contrib = xh_t_full[incoming_ho]  # (M, d_out)
fo_term_v = (deg_full[target_node] * xf_t_full[target_node]).detach()  # (d_out,)

# Full out vector at v (should equal contrib.sum + fo_term_v)
out_full_v = (out_full[target_node]).detach()

# --- Explainer score: margin drop when removing HO-node contribution ---
scores_margin_drop = np.zeros(M, dtype=float)
m_full = margin_from_out_v(out_full_v, y_ref)

for i in range(M):
    out_minus = out_full_v - contrib[i]
    m_minus = margin_from_out_v(out_minus, y_ref)
    scores_margin_drop[i] = m_full - m_minus

# --- Model-ground-truth: Shapley values (permutation approximation) ---
rng = np.random.default_rng(0)
phi = np.zeros(M, dtype=float)

# baseline: no HO contributions (sum = 0), but keep FO term (deg is constant)
def f_from_sum(sum_vec: torch.Tensor):
    return margin_from_out_v(sum_vec + fo_term_v, y_ref)

for _ in range(lvl1_permutations):
    perm = rng.permutation(M)
    sum_vec = torch.zeros_like(fo_term_v)
    f_prev = f_from_sum(sum_vec)
    for idx in perm:
        sum_vec = sum_vec + contrib[idx]
        f_new = f_from_sum(sum_vec)
        phi[idx] += (f_new - f_prev)
        f_prev = f_new

phi /= float(lvl1_permutations)

# --- Generator-ground-truth: "signal" if y[u]==y[v] ---
u_list = incoming_uv[:, 0]
signal_lvl1 = (y[u_list] == y[target_node]).astype(int)

# --- Metrics ---
def spearman_corr(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    ra = a.argsort().argsort().astype(float)
    rb = b.argsort().argsort().astype(float)
    ra -= ra.mean()
    rb -= rb.mean()
    denom = (np.sqrt((ra**2).sum()) * np.sqrt((rb**2).sum()) + 1e-12)
    return float((ra * rb).sum() / denom)

def ndcg_at_k(scores, rel, k=10):
    # scores: ranking scores (higher=more important)
    # rel: ground-truth relevance (non-negative)
    k = min(k, len(scores))
    order = np.argsort(scores)[::-1][:k]
    rel_sorted = rel[order]
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    dcg = float((rel_sorted * discounts).sum())
    # ideal
    ideal = np.sort(rel)[::-1][:k]
    idcg = float((ideal * discounts).sum()) + 1e-12
    return dcg / idcg

# Relevance for NDCG: use positive Shapley only (negative = not helpful to class y_ref)
rel = np.maximum(phi, 0.0)

spearman = spearman_corr(scores_margin_drop, phi)
ndcg10 = ndcg_at_k(scores_margin_drop, rel, k=10)

print("\n--- Level 1: Model-ground-truth (Shapley) ---")
print("Spearman(explainer, Shapley):", spearman)
print("NDCG@10 (explainer vs positive Shapley):", ndcg10)

# Generator-ground-truth metrics
if len(np.unique(signal_lvl1)) > 1:
    auc = roc_auc_score(signal_lvl1, scores_margin_drop)
else:
    auc = float("nan")

k_eval = min(10, M)
topk = np.argsort(scores_margin_drop)[::-1][:k_eval]
prec_at_k = float(signal_lvl1[topk].mean())

print("\n--- Level 1: Generator-ground-truth (within-cluster u->v) ---")
print("AUROC(signal, explainer_score):", auc)
print(f"Precision@{k_eval}:", prec_at_k)

# Show top incoming HO nodes by explainer score
print("\nTop incoming HO nodes (u -> v) by margin-drop score:")
for rank, idx in enumerate(topk[:min(k_eval, 10)], start=1):
    ho_id = int(incoming_ho[idx])
    u, v = map(int, incoming_uv[idx])
    print(f"{rank:2d}. HO_node_idx={ho_id:4d}  (u,v)=({u},{v})  score={scores_margin_drop[idx]:.6f}  signal={signal_lvl1[idx]}")


In [ ]:

# ============================================================
# Level 2 evaluation: HO edges (u,v)->(v,w) for important HO nodes
# ============================================================

import numpy as np
import pandas as pd

# Collect Level-2 results for Plotly visualizations
lvl2_edge_rows = []
lvl2_summary_rows = []

# Choose HO nodes to explain (top_m incoming HO nodes by Level 1 explainer score)
order_lvl1 = np.argsort(scores_margin_drop)[::-1]
selected = order_lvl1[:min(top_m, M)]

print("\nSelected HO nodes for Level 2 (top_m={}):".format(min(top_m, M)))
for j, idx in enumerate(selected, start=1):
    ho_id = int(incoming_ho[idx])
    u, v = map(int, incoming_uv[idx])
    print(f"{j:2d}. HO_node_idx={ho_id:4d} (u,v)=({u},{v})  lvl1_score={scores_margin_drop[idx]:.6f}")

def cosine(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-12):
    return float((a @ b) / ((a.norm() * b.norm()) + eps))

for idx in selected:
    q = int(incoming_ho[idx])  # HO node index
    u_q, v_q = map(int, g2_node_ids[q])
    assert v_q == target_node, "Selected HO node should map to target_node."

    # Build k-hop induced subgraph around q
    node_set = k_hop_nodes(q, k_hop, direction="both")
    edge_ids = edges_induced_by_nodes(node_set)
    Ek = len(edge_ids)

    print("\n" + "-" * 70)
    print(f"Level 2 for HO node q={q} corresponding to (u,v)=({u_q},{v_q})")
    print(f"k-hop nodes: {len(node_set)} | induced HO edges: {Ek}")

    if Ek == 0:
        print("No edges in k-hop induced subgraph -> skipping.")
        continue

    edge_ids_t = torch.tensor(edge_ids, dtype=torch.long, device=device)

    # Full mask for induced edges only
    full_mask = torch.zeros(E_ho, device=device)
    full_mask[edge_ids_t] = 1.0

    # Compute baseline + full (restricted) embeddings for q
    with torch.no_grad():
        H_base = ho_embed(torch.zeros_like(edge_weight_ho))
        h_base_q = H_base[q].detach()

        H_local_full = ho_embed(edge_weight_ho * full_mask)
        h_local_full_q = H_local_full[q].detach()

        h_global_full_q = H_full[q].detach()

        delta = (h_local_full_q - h_base_q)
        delta_norm = float(delta.norm().item())

        cos_local_global = cosine(h_local_full_q, h_global_full_q)
        rel_diff = float((h_local_full_q - h_global_full_q).norm().item() / (h_global_full_q.norm().item() + 1e-12))

    print(f"Embedding delta norm (local full vs base): {delta_norm:.6f}")
    print(f"Local-vs-global HO embedding cos: {cos_local_global:.4f} | rel L2 diff: {rel_diff:.4f}")

    if delta_norm < 1e-6:
        print("Delta ~ 0, HO edges in this neighborhood barely change H_q -> skipping IG/LOO.")
        continue

    # ------------------------------------------------------------
    # Explainer (Level 2): Integrated Gradients on edge-mask m in [0,1]
    # Objective: s(m) = (h_q(m)-h_base_q) · delta
    # ------------------------------------------------------------
    accum = torch.zeros(Ek, device=device)
    for t in range(ig_steps):
        alpha = (t + 0.5) / ig_steps
        mask_params = torch.full((Ek,), float(alpha), device=device, requires_grad=True)

        m = torch.zeros(E_ho, device=device)
        m[edge_ids_t] = mask_params
        w = edge_weight_ho * m

        H_alpha = ho_embed(w)
        h_alpha = H_alpha[q]
        s = (h_alpha - h_base_q).dot(delta)

        grad = torch.autograd.grad(s, mask_params, retain_graph=False, create_graph=False)[0]
        accum += grad.detach()

    ig_scores = (accum / ig_steps).detach().cpu().numpy()  # shape (Ek,)

    # Rank edges by absolute IG (you can also use positive-only)
    rank_ig = np.argsort(np.abs(ig_scores))[::-1]

    # ------------------------------------------------------------
    # Model-ground-truth (Level 2): Leave-one-out (LOO) on same objective
    # ------------------------------------------------------------
    # scalar_full = (h_full - h_base)·delta = ||delta||^2
    scalar_full = float(delta.dot(delta).item())
    loo = np.zeros(Ek, dtype=float)

    # pre-build "ones" mask params
    ones = torch.ones(Ek, device=device)
    with torch.no_grad():
        # we already have h_base_q and delta and scalar_full
        pass

    # Evaluate LOO for each edge (can be slow if Ek is large)
    for j in range(Ek):
        mask_params = ones.clone()
        mask_params[j] = 0.0
        m = torch.zeros(E_ho, device=device)
        m[edge_ids_t] = mask_params
        w = edge_weight_ho * m
        with torch.no_grad():
            H_minus = ho_embed(w)
            h_minus = H_minus[q]
            scalar_minus = float((h_minus - h_base_q).dot(delta).item())
        loo[j] = scalar_full - scalar_minus

    # Compare IG ranking to LOO ranking
    spearman2 = spearman_corr(np.abs(ig_scores), loo)
    rel2 = np.maximum(loo, 0.0)
    ndcg2 = ndcg_at_k(np.abs(ig_scores), rel2, k=min(10, Ek))

    print("\n--- Level 2: Model-ground-truth (LOO) ---")
    print("Spearman(|IG|, LOO):", spearman2)
    print("NDCG@10 (|IG| vs positive LOO):", ndcg2)

    # ------------------------------------------------------------
    # Generator-ground-truth for HO edges: signal if y[u]==y[v]==y[w]
    # For each HO edge e: src HO node = (a,b), dst HO node = (b,c)
    # ------------------------------------------------------------
    # Extract triples for edges in this induced set
    src_nodes = src[edge_ids]
    dst_nodes = dst[edge_ids]
    a = g2_node_ids[src_nodes, 0]
    b = g2_node_ids[src_nodes, 1]
    c = g2_node_ids[dst_nodes, 1]
    signal_lvl2 = ((y[a] == y[b]) & (y[b] == y[c])).astype(int)

    # Store per-edge rows (for Plotly)
    for jj in range(Ek):
        e_id = int(edge_ids[jj])
        s_node = int(src[e_id])
        d_node = int(dst[e_id])
        u1, v1 = map(int, g2_node_ids[s_node])
        w2 = int(g2_node_ids[d_node, 1])
        lvl2_edge_rows.append({
            "q": int(q),
            "u_q": int(u_q),
            "v_q": int(v_q),
            "edge_local_idx": int(jj),
            "edge_id": int(e_id),
            "u": int(u1),
            "v": int(v1),
            "w": int(w2),
            "ig": float(ig_scores[jj]),
            "abs_ig": float(abs(ig_scores[jj])),
            "loo": float(loo[jj]),
            "signal": int(signal_lvl2[jj]),
            "k_hop": int(k_hop),
            "Ek": int(Ek),
            "node_set_size": int(len(node_set)),
        })

    if len(np.unique(signal_lvl2)) > 1:
        auc2 = roc_auc_score(signal_lvl2, np.abs(ig_scores))
    else:
        auc2 = float("nan")

    k_eval = min(top_e, Ek)
    topk_edges = rank_ig[:k_eval]
    prec2 = float(signal_lvl2[topk_edges].mean())

    # Store per-HO-node summary (for Plotly)
    lvl2_summary_rows.append({
        "q": int(q),
        "u_q": int(u_q),
        "v_q": int(v_q),
        "k_hop": int(k_hop),
        "node_set_size": int(len(node_set)),
        "Ek": int(Ek),
        "delta_norm": float(delta_norm),
        "cos_local_global": float(cos_local_global),
        "rel_diff_local_global": float(rel_diff),
        "spearman_absIG_LOO": float(spearman2),
        "ndcg10_absIG_vs_posLOO": float(ndcg2),
        "auc_signal_absIG": float(auc2),
        "prec_at_k": float(prec2),
        "k_eval": int(k_eval),
    })

    print("\n--- Level 2: Generator-ground-truth (within-cluster u->v->w) ---")
    print("AUROC(signal, |IG|):", auc2)
    print(f"Precision@{k_eval}:", prec2)

    print("\nTop HO edges by |IG|:")
    for r in range(min(top_e, Ek)):
        j = topk_edges[r]
        e_id = int(edge_ids[j])
        s_node = int(src[e_id])
        d_node = int(dst[e_id])
        u1, v1 = map(int, g2_node_ids[s_node])
        v2, w2 = map(int, g2_node_ids[d_node])
        print(f"{r+1:2d}. edge_id={e_id:4d}  ({u1},{v1})->({v2},{w2})  |IG|={abs(ig_scores[j]):.6f}  LOO={loo[j]:.6f}  signal={signal_lvl2[j]}")

# --- Collect results into DataFrames for Plotly ---
lvl2_edges_df = pd.DataFrame(lvl2_edge_rows)
lvl2_summary_df = pd.DataFrame(lvl2_summary_rows)

print("\nLevel 2 edges table (first 5 rows):")
print(lvl2_edges_df.head())


# Visualizations (Plotly)

The following plots help you *visually* inspect both ground-truth notions:

- **Model-ground-truth (faithfulness):** comparison of explainer scores to **Shapley (Level 1)** and **LOO (Level 2)**.
- **Generator-ground-truth (planted mechanism):** whether top-ranked items are mostly **within-cluster**.

These are interactive Plotly plots (hover to see edge/node IDs).


In [ ]:
! pip install plotly

In [ ]:
# Plotly visualizations — Level 1 (incoming HO nodes for target_node)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.metrics import roc_auc_score, roc_curve

pio.renderers.default = "notebook_connected"

df_lvl1 = pd.DataFrame({
    "ho_node_idx": incoming_ho.astype(int),
    "u": incoming_uv[:, 0].astype(int),
    "v": incoming_uv[:, 1].astype(int),
    "margin_drop": scores_margin_drop.astype(float),
    "shapley": phi.astype(float),
    "signal": signal_lvl1.astype(int),
})
df_lvl1["signal_label"] = df_lvl1["signal"].map({0: "cross-cluster", 1: "within-cluster"})

# 1) Scatter: Shapley vs margin-drop (faithfulness check)
fig = px.scatter(
    df_lvl1,
    x="shapley",
    y="margin_drop",
    color="signal_label",
    hover_data=["ho_node_idx", "u", "v", "margin_drop", "shapley"],
    title=f"Level 1 (node v={target_node}): Shapley vs margin-drop for incoming HO nodes (u,v)",
)
fig.show()

# 2) Grouped bars: top-k by explainer score
k_show = min(10, len(df_lvl1))
df_top = df_lvl1.sort_values("margin_drop", ascending=False).head(k_show).copy()
df_top["label"] = df_top.apply(lambda r: f"{int(r.u)}→{int(r.v)} (HO {int(r.ho_node_idx)})", axis=1)
df_top = df_top.iloc[::-1]  # reverse for nicer horizontal bars

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=df_top["margin_drop"],
    y=df_top["label"],
    orientation="h",
    name="Explainer (margin drop)",
))
fig2.add_trace(go.Bar(
    x=df_top["shapley"],
    y=df_top["label"],
    orientation="h",
    name="Model-ground-truth (Shapley)",
))
fig2.update_layout(
    barmode="group",
    title=f"Level 1 top-{k_show}: explainer vs Shapley",
    xaxis_title="score",
    yaxis_title="incoming HO node (u→v)",
)
fig2.show()

# 3) ROC curve: generator-ground-truth (within-cluster u->v)
if len(np.unique(signal_lvl1)) > 1:
    auc_plot = roc_auc_score(signal_lvl1, scores_margin_drop)
    fpr, tpr, _ = roc_curve(signal_lvl1, scores_margin_drop)

    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines", name="Explainer"))
    fig3.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Random"))
    fig3.update_layout(
        title=f"Level 1 ROC (signal=within-cluster u→v) | AUC={auc_plot:.3f}",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate",
    )
    fig3.show()
else:
    print("Level 1 ROC not available: signal_lvl1 has only one class.")

# 4) Deletion curve: margin as we remove incoming HO nodes in explainer order
order = np.argsort(scores_margin_drop)[::-1]
out_cur = out_full_v.clone()
margins = [margin_from_out_v(out_cur, y_ref)]
for t in order:
    out_cur = out_cur - contrib[t]
    margins.append(margin_from_out_v(out_cur, y_ref))

# Random baseline (mean over random permutations)
rng = np.random.default_rng(0)
R = 200
rand_marg = np.zeros((R, M + 1), dtype=float)
for r in range(R):
    perm = rng.permutation(M)
    out_r = out_full_v.clone()
    rand_marg[r, 0] = margin_from_out_v(out_r, y_ref)
    for i, p in enumerate(perm, start=1):
        out_r = out_r - contrib[p]
        rand_marg[r, i] = margin_from_out_v(out_r, y_ref)
rand_mean = rand_marg.mean(axis=0)

x = list(range(M + 1))
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=x, y=margins, mode="lines+markers", name="Remove in explainer order"))
fig4.add_trace(go.Scatter(x=x, y=rand_mean, mode="lines", name="Remove in random order (mean)"))
fig4.update_layout(
    title=f"Level 1 deletion curve (node v={target_node}): margin vs #removed incoming HO nodes",
    xaxis_title="# removed incoming HO nodes",
    yaxis_title="margin",
)
fig4.show()


In [ ]:
# Plotly visualizations — Level 2 (HO edges explaining HO embeddings)

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

assert "lvl2_edges_df" in globals(), "Run the Level 2 evaluation cell first (it creates lvl2_edges_df)."
assert len(lvl2_edges_df) > 0, "lvl2_edges_df is empty — try increasing top_m or k_hop and re-run Level 2."

df = lvl2_edges_df.copy()
df["signal_label"] = df["signal"].map({0: "cross-cluster", 1: "within-cluster"})
df["edge_str"] = df.apply(lambda r: f"({int(r.u)},{int(r.v)})→({int(r.v)},{int(r.w)}) [id {int(r.edge_id)}]", axis=1)

# 1) Scatter: |IG| vs LOO (faithfulness check), faceted by HO node q
fig = px.scatter(
    df,
    x="loo",
    y="abs_ig",
    color="signal_label",
    facet_col="q",
    facet_col_wrap=2,
    hover_data=["edge_id", "u", "v", "w", "abs_ig", "loo", "signal_label"],
    title="Level 2: |IG| vs LOO for HO edges in the k-hop neighborhood (faceted by HO node q)",
)
fig.show()

# 2) Top edges per q by |IG| (generator signal shown by color)
top_e_show = min(int(top_e), 15)
df_top = df.sort_values("abs_ig", ascending=False).groupby("q", group_keys=False).head(top_e_show)

fig2 = px.bar(
    df_top,
    x="abs_ig",
    y="edge_str",
    color="signal_label",
    facet_col="q",
    facet_col_wrap=2,
    orientation="h",
    title=f"Level 2: Top-{top_e_show} HO edges by |IG| (per HO node q)",
)
fig2.show()

# 3) Neighborhood fidelity summary (how well the k-hop subgraph matches the global embedding)
if "lvl2_summary_df" in globals() and len(lvl2_summary_df) > 0:
    fig3 = px.scatter(
        lvl2_summary_df,
        x="Ek",
        y="cos_local_global",
        size="delta_norm",
        hover_data=["q", "node_set_size", "rel_diff_local_global"],
        title="Level 2: k-hop neighborhood fidelity for HO embedding (cos(local,global); size=delta_norm)",
    )
    fig3.show()

    # Bar: AUROC & Precision@k per q (generator-ground-truth)
    fig4 = px.bar(
        lvl2_summary_df,
        x="q",
        y=["auc_signal_absIG", "prec_at_k"],
        barmode="group",
        title="Level 2 generator-ground-truth per HO node q (AUROC and Precision@k)",
    )
    fig4.show()
else:
    print("lvl2_summary_df not found or empty.")
